# IND320, Assignment 3

## By: Ole Andreas Grøtting
Github: [IND320 - OLEGRO99](https://github.com/olegro99/IND320-OLEGRO99.git)\
Streamlit: https://ind320-olegro99-ybwpxf3ssrwyekrthttcuc.streamlit.app/

## Imports

In [2]:
import geopandas as gpd
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [4]:
#Read the shapefile from atlas.nve.no
#include all files
gdf = gpd.read_file("../NVE_Region.shp")

#generate K random points within each region
K = 100  # Number of random points to generate per region
points_per_region = gdf['geometry'].sample_points(size=K, random_state=42)

# Transform WGS84 for latitude and longitude
points_wgs84 = points_per_region.to_crs(epsg=4326)

#extract the coordinates and print them
for idx, multipoint in points_wgs84.items():
    region_name = gdf.loc[idx, 'REGION']
    print(f"Region: {region_name}")
    lat = ""
    lon = ""
    for point in multipoint.geoms:
        # Do something more useful with the coordinates
        lat = lat + ","+ str(point.y)
        lon = lon + ","+ str(point.x)

gdf["AREA"] = gdf["REGION"].replace({
    "RS": "East",
    "RØ": "East",
    "RM": "North",
    "RN": "North"
})


Region: RV
Region: RM
Region: RN
Region: RÃ
Region: RS


In [9]:
lat[1:]

'59.38747399132355,59.49929420352692,58.74491057259957,60.396922209789246,58.95378498300207,58.44578217909316,58.567152142688165,59.0779009978686,58.133513116583835,60.63408403214976,59.71334747731769,58.742360830162,58.63787114953937,58.307126643748106,59.47938129468181,58.422404179334656,59.11747238308827,58.488675029521644,58.89739165964447,59.90673217553659,57.94997232518531,59.56214599039743,59.31558416420555,59.050091824168135,59.32498505055129,60.00223384047867,60.354903107899425,59.21187309848325,59.88557555512143,59.81985606698738,58.576986493485485,58.68302732359218,58.5627825928464,59.12428294088199,58.218958091174486,60.34250058338102,58.81389309159246,59.69026911161932,58.805826340495216,60.81339378041196,60.76358111525722,58.89178859960178,59.03561761087639,60.770139618532305,59.192573730410814,59.97982429068922,58.45152305291135,59.2020509475117,57.82910006210531,59.663528536443366,59.34074167514283,60.0203059080968,59.15601694711486,58.821421552505136,58.15971150959745,

In [5]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

openmeteo = openmeteo_requests.Client(session=retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat[1:],
    "longitude": lon[1:],
    "start_date": "2026-10-12",
    "end_date": "2026-10-26",
    "hourly": ["temperature_2m", "temperature_2m_spread", "wind_speed_10m_spread"],
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for loop for multiple locations or weather models
for i in range(len(responses)):
    response = responses[i]
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    print("\nHourly temperature_2m\n", hourly_temperature_2m)


Hourly temperature_2m
 [ 7.15  7.15  7.2   7.3   7.05  6.9   6.95  6.95  7.25  7.45  7.45  7.75
  8.    8.25  8.5   8.5   8.75  9.15  9.55  9.9  10.05 10.15 10.35 10.5
 10.6  10.85 11.1  11.   11.1  11.1  11.05 11.2  11.3  11.35 11.4  11.4
 11.45 11.7  11.5  11.85 12.3  12.55 12.75 13.   13.7  12.7  12.6  12.35
 11.9  11.85 11.8  11.75 11.75 11.55 11.1  10.95 11.25 11.35 11.7  12.3
 13.25 13.45 13.5  13.45 13.3  13.05 12.8  12.55 12.7  13.1  13.3  13.4
 13.55 13.6  13.7  13.7  13.8  13.9  14.   13.95 14.   14.   14.   14.
 13.9  14.1  13.95 14.   13.95 13.95 13.9  13.8  13.75 13.75 13.65 13.7
 13.7  13.5  13.35 12.95 12.3  11.75 11.8  11.9  12.05 11.9  11.3  10.85
 10.75 10.75 10.95 11.3  11.6  11.65 11.9  11.25 10.65 10.45 10.1  10.35
 10.4  10.25 10.1  10.1  10.2  10.    9.95  9.75  9.75  9.85 10.   10.2
 10.55 10.9  11.2  11.5  11.65 11.8  11.7  11.35 10.9  10.65 10.15  9.7
  9.45  9.3   9.5   9.6   9.75  9.75  9.8   9.9   9.75  9.25  9.45  9.65
  9.75  9.8  10.   10.   10.3  10.6 